# Exp 7 - Wireless Communication Delay Analysis for IEEE 802.11, LTE, and 5G

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Compare delay and jitter behaviour for three wireless communication profiles used around autonomous systems.

The notebook uses synthetic profiles for IEEE 802.11, LTE, and 5G to show how mean delay, jitter, and P95 delay affect control and telemetry suitability.

## Textbook Notes and Case Studies

### 1. Textbook Background

Wireless communication delay depends on propagation, medium access, scheduling, retransmission, signal quality, and core-network processing. IEEE 802.11, LTE, and 5G have different delay behavior because their access methods and network architectures differ.

Wi-Fi can provide high data rates, but contention and interference can increase jitter. LTE uses cellular scheduling through a base station, which provides managed access but adds network scheduling delay. 5G is designed to support lower-latency services through architectural and radio improvements, but real performance still depends on deployment, load, coverage, and configuration.

### 2. Architecture Notes

```
Vehicle Device -> Wireless Access Link -> Access Point / Base Station -> Network/Core -> Edge or Cloud Service
       |                  |                         |                       |
       v                  v                         v                       v
Radio Conditions     MAC Delay                Scheduling Delay          Processing Delay
```

The experiment compares modeled delay components. A realistic analysis should separate radio access delay from backhaul and processing delay. Otherwise, the wrong part of the system may be optimized.

### 3. Important Formulas

Total delay model:

```
total_delay = access_delay + propagation_delay + queueing_delay + processing_delay
```

Mean delay:

```
mean_delay = sum(delay_samples) / n
```

Jitter:

```
jitter = variation in delay across samples
```

Deadline violation:

```
violation = total_delay > application_deadline
```

### 4. Classroom Case Studies

Case Study A - Campus Shuttle Wi-Fi:
An autonomous shuttle uses Wi-Fi near a depot. Delay is low when the channel is clear, but contention from many users increases jitter during peak periods.

Case Study B - LTE Telemetry:
A fleet uploads telemetry through LTE. Managed cellular coverage provides wide-area connectivity, but telemetry may not be appropriate for tight control loops if delay budgets are strict.

Case Study C - 5G Roadside Edge:
A 5G roadside edge application processes hazard warnings close to the road. The edge location reduces backhaul delay, but the system still needs measured latency under load before safety claims can be made.

### 5. Analysis Checklist

Compare mean, maximum, and deadline-violation count for each technology. Avoid writing that one technology is always best. The correct conclusion depends on environment, load, distance, deployment, and application deadline.

### 6. Source Notes

- IEEE 802 networking standards background: https://www.ieee802.org/
- 3GPP specifications are the primary standards source for LTE and 5G systems: https://www.3gpp.org/specifications


## Architecture

```text
Wireless Technology Profile
  |-- baseline delay
  |-- variation/spread
          |
          v
Packet Delay Generator
          |
          v
Timing Analyzer
  |-- mean delay
  |-- average jitter
  |-- P95 delay
          |
          v
Technology Comparison
```

## Formulas and Required Theory

\[
\text{jitter}_i = |delay_i - delay_{i-1}|
\]

\[
\text{P95 delay} = \text{delay below which 95% of samples fall}
\]

Mobility penalty used in the post-lab model:

\[
delay_{tech}(v) = delay_{base} + k_{tech} \times v
\]

where \(v\) is vehicle speed in km/h and \(k_{tech}\) is a technology-specific penalty factor.

## In-Lab Method

1. Define representative delay profiles for IEEE 802.11, LTE, and 5G.
2. Generate delay samples for each technology.
3. Compute mean delay, average jitter, and P95 delay.
4. Compare which profile is more suitable for low-latency autonomous communication.

In [1]:
import random
import statistics

profiles = {
    "IEEE 802.11": (18, 6),
    "LTE": (28, 8),
    "5G": (8, 3),
}
rng = random.Random(341407)
print("EXP 7 - IN-LAB WIRELESS DELAY")
print(f"{'Technology':12} {'Mean ms':>8} {'Jitter ms':>10} {'P95 ms':>8}")
for tech, (base, spread) in profiles.items():
    samples = [max(1, rng.gauss(base, spread)) for _ in range(80)]
    jitter = [abs(samples[i] - samples[i-1]) for i in range(1, len(samples))]
    p95 = sorted(samples)[int(0.95 * (len(samples) - 1))]
    print(f"{tech:12} {statistics.mean(samples):8.2f} {statistics.mean(jitter):10.2f} {p95:8.2f}")

EXP 7 - IN-LAB WIRELESS DELAY
Technology    Mean ms  Jitter ms   P95 ms
IEEE 802.11     17.27       6.91    25.42
LTE             28.26       9.15    42.35
5G               7.87       3.33    12.19


## Post-Lab Method

The post-lab cell adds a mobility penalty as vehicle speed increases. This demonstrates that a wireless link must be evaluated under motion, not only in a static condition.

In [2]:
print("EXP 7 - POST-LAB MOBILITY PENALTY")
for speed in [0, 30, 60, 90, 120]:
    wifi = 18 + speed * 0.08
    lte = 28 + speed * 0.03
    fiveg = 8 + speed * 0.02
    print(f"speed={speed:3} km/h | 802.11={wifi:5.1f} ms | LTE={lte:5.1f} ms | 5G={fiveg:5.1f} ms")
print("Inference: 5G keeps the smallest delay in this model; Wi-Fi degrades faster with mobility.")

EXP 7 - POST-LAB MOBILITY PENALTY
speed=  0 km/h | 802.11= 18.0 ms | LTE= 28.0 ms | 5G=  8.0 ms
speed= 30 km/h | 802.11= 20.4 ms | LTE= 28.9 ms | 5G=  8.6 ms
speed= 60 km/h | 802.11= 22.8 ms | LTE= 29.8 ms | 5G=  9.2 ms
speed= 90 km/h | 802.11= 25.2 ms | LTE= 30.7 ms | 5G=  9.8 ms
speed=120 km/h | 802.11= 27.6 ms | LTE= 31.6 ms | 5G= 10.4 ms
Inference: 5G keeps the smallest delay in this model; Wi-Fi degrades faster with mobility.


## What to Write in the Lab Record

- Include the delay comparison table.
- Identify the technology with the lowest mean delay and P95 delay.
- Explain why mobility can increase timing variation.
- State that real results require physical measurement or a validated network simulator.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html